In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_catalog.gold;

USE CATALOG retail_catalog;
USE SCHEMA gold;

In [0]:
from pyspark.sql.functions import *

df = spark.read.table("retail_catalog.silver.inventory_conformed")

In [0]:
##  IN AND OUT MOV BASED ON CATEGORY 
kpi_category_movement = df.groupBy("category") \
    .agg(
        sum(when(col("movement_type")=="IN", col("quantity"))).alias("total_in"),
        sum(when(col("movement_type")=="OUT", col("quantity"))).alias("total_out")
    )

display(kpi_category_movement)

category,total_in,total_out
SPORTS,1468193,1364074
ELECTRONICS,1460074,1340693
APPAREL,1451132,1358946
HOME,1451769,1370020
GROCERY,1436011,1358410


Databricks visualization. Run in Databricks to view.

In [0]:
## CURRENT QUANTITY BASED ON WH LOCATION 
kpi_warehouse = df.groupBy("location") \
    .agg(sum("quantity").alias("total_quantity"))

display(kpi_warehouse)

location,total_quantity
Bangalore,3887180
Mumbai,4385701
Hyderabad,2751101
Delhi,1096796
Chennai,1938544


Databricks visualization. Run in Databricks to view.

In [0]:
##inventory value by each category
kpi_value = df.groupBy("category") \
    .agg(sum(col("quantity") * col("unit_cost")).alias("inventory_value"))

display(kpi_value)

category,inventory_value
SPORTS,3.794909343220006E9
ELECTRONICS,9.820678415990072E9
APPAREL,2.4112472540199704E9
HOME,1.7864459819400039E9
GROCERY,1.7085243671999955E8


Databricks visualization. Run in Databricks to view.

In [0]:
##top 10 sku by quantity
top_sku = (
    df.groupBy("sku_id")
      .agg(sum("quantity").alias("total_qty"))
      .orderBy(col("total_qty").desc())
      .limit(10)  
)

display(top_sku)

sku_id,total_qty
SKU1564,23572
SKU1938,22496
SKU1955,22044
SKU1496,21934
SKU1751,21651
SKU1615,21402
SKU1853,21293
SKU1258,21249
SKU1870,21214
SKU1020,21189


Databricks visualization. Run in Databricks to view.

In [0]:
df = df.withColumn(
    "month",
    date_format(col("event_date"), "yyyy-MM")
)

In [0]:
##MONTHLY INVENTORY BALANCE 
monthly_inventory_balance = df.withColumn(
    "net_qty",
    when(col("movement_type") == "IN", col("quantity"))
    .otherwise(-col("quantity"))
).groupBy("month").agg(
    sum("net_qty").alias("net_inventory")
).orderBy("month")

display(monthly_inventory_balance)

month,net_inventory
2026-01,89660
2026-02,83641
2026-03,217949
2026-04,81756
2026-05,2030


Databricks visualization. Run in Databricks to view.

In [0]:
##daily in and out movement trend 
kpi_daily = df.groupBy("event_date") \
    .agg(
        sum(when(col("movement_type")=="IN", col("quantity"))).alias("total_in"),
        sum(when(col("movement_type")=="OUT", col("quantity"))).alias("total_out")
    )

display(kpi_daily)

event_date,total_in,total_out
2026-01-19,47366,40190
2026-03-02,1864860,1739693
2026-02-10,42395,42306
2026-03-10,41246,41078
2026-03-25,48761,42304
2026-03-21,44499,41762
2026-04-22,41740,41857
2026-03-03,49518,43235
2026-04-09,48936,42074
2026-02-05,47676,38088


Databricks visualization. Run in Databricks to view.

In [0]:
## total inventory balance by each location and sku 
from pyspark.sql.functions import *

# STEP 1: Convert IN/OUT to net movement
df_balance = (
    df.withColumn(
        "net_qty",
        when(col("movement_type") == "IN", col("quantity"))
        .otherwise(-col("quantity"))
    )
)

# STEP 2: Calculate current inventory per SKU + warehouse
current_inventory = (
    df_balance
    .groupBy("sku_id", "warehouse_id")
    .agg(sum("net_qty").alias("current_stock"))
    .filter(col("current_stock") > 0)   # only valid stock
)

# STEP 3: Add location & category (business context)
current_inventory = current_inventory \
    .join(df.select("warehouse_id", "location").distinct(), "warehouse_id", "left") \
    .join(df.select("sku_id", "category").distinct(), "sku_id", "left")


top_inventory = (
    current_inventory
    .orderBy(col("current_stock").desc())
    .limit(10)
)

display(top_inventory)

sku_id,warehouse_id,current_stock,location,category
SKU1978,WH01,2365,Mumbai,SPORTS
SKU1408,WH01,2104,Mumbai,SPORTS
SKU1012,WH01,2101,Mumbai,APPAREL
SKU1971,WH01,2074,Mumbai,ELECTRONICS
SKU1489,WH01,2073,Mumbai,HOME
SKU1953,WH01,1984,Mumbai,SPORTS
SKU1007,WH01,1971,Mumbai,APPAREL
SKU1653,WH02,1948,Bangalore,SPORTS
SKU1350,WH02,1940,Bangalore,GROCERY
SKU1085,WH01,1932,Mumbai,GROCERY


Databricks visualization. Run in Databricks to view.

In [0]:
## total quantity by each category
kpi_category_qty = df.groupBy("category") \
    .agg(sum("quantity").alias("total_quantity"))

display(kpi_category_qty)

category,total_quantity
SPORTS,2832267
ELECTRONICS,2800767
APPAREL,2810078
HOME,2821789
GROCERY,2794421


Databricks visualization. Run in Databricks to view.

In [0]:
## Daily snapshot of each category

from pyspark.sql.functions import *
from pyspark.sql.window import Window

# -------------------------
# STEP 0: Clean base data
# -------------------------
df = df.withColumn("sku_id", trim(col("sku_id")))

df = df.withColumn(
    "category",
    when(col("category").isNull(), lit("UNMAPPED"))
    .otherwise(col("category"))
)

# -------------------------
# STEP 1: Filter last 30 days
# -------------------------
df_filtered = (
    df.withColumn("event_date", to_date(col("event_time")))
      .filter(col("event_date") >= date_sub(current_date(), 30))
)

# -------------------------
# STEP 2: Daily movement
# -------------------------
df_daily = (
    df_filtered
    .withColumn(
        "net_qty",
        when(col("movement_type") == "IN", col("quantity"))
        .otherwise(-col("quantity"))
    )
    .groupBy("sku_id", "event_date")
    .agg(sum("net_qty").alias("daily_movement"))
)

# -------------------------
# STEP 3: Cumulative stock
# -------------------------
window = Window.partitionBy("sku_id").orderBy("event_date")

df_snapshot = (
    df_daily
    .withColumn("closing_stock", sum("daily_movement").over(window))
)

# -------------------------
# STEP 4: Join category
# -------------------------
df_category = df.select("sku_id", "category").distinct()

df_snapshot = df_snapshot.join(
    df_category,
    "sku_id",
    "left"
)

# -------------------------
# STEP 5: Fix category after join
# -------------------------
df_snapshot = df_snapshot.withColumn(
    "category",
    when(col("category").isNull(), lit("UNMAPPED"))
    .otherwise(col("category"))
)

# -------------------------
# STEP 6: Category-level stock
# -------------------------
category_snapshot = (
    df_snapshot
    .groupBy("event_date", "category")
    .agg(sum("closing_stock").alias("total_stock"))
)

# -------------------------
# STEP 7: Remove unwanted values
# -------------------------
category_snapshot = category_snapshot.filter(
    (col("total_stock") > 0) & 
    (col("category").isNotNull()) &
    (col("category") != "UNMAPPED")
)

# -------------------------
# FINAL OUTPUT
# -------------------------
display(category_snapshot)

event_date,category,total_stock
2026-04-07,GROCERY,2741
2026-04-08,GROCERY,219
2026-04-09,GROCERY,868
2026-04-10,GROCERY,705
2026-04-11,GROCERY,1690
2026-04-12,GROCERY,2143
2026-04-14,GROCERY,2177
2026-04-15,GROCERY,7859
2026-04-16,GROCERY,4593
2026-04-17,GROCERY,10125


Databricks visualization. Run in Databricks to view.

In [0]:
category_snapshot.write.mode("overwrite") \
    .saveAsTable("retail_catalog.gold.category_snapshot")
top_sku.write.mode("overwrite") \
    .saveAsTable("retail_catalog.gold.top_sku")
kpi_category_movement.write.mode("overwrite") \
    .saveAsTable("retail_catalog.gold.kpi_category_movement")
kpi_warehouse.write.mode("overwrite") \
    .saveAsTable("retail_catalog.gold.kpi_warehouse")
kpi_value.write.mode("overwrite") \
    .saveAsTable("retail_catalog.gold.kpi_value")
kpi_daily.write.mode("overwrite") \
    .saveAsTable("retail_catalog.gold.kpi_daily")
monthly_inventory_balance.write.mode("overwrite") \
    .saveAsTable("retail_catalog.gold.monthly_inventory_balance")
top_inventory.write.mode("overwrite") \
    .saveAsTable("retail_catalog.gold.top_inventory")
kpi_category_qty.write.mode("overwrite") \
    .saveAsTable("retail_catalog.gold.kpi_category_qty")


In [0]:
%sql
SELECT
    -- 📦 Total Current Stock
    SUM(on_hand_qty) AS total_current_stock,

    -- 💰 Total Inventory Value (₹)
    ROUND(SUM(on_hand_qty * unit_cost), 2) AS total_inventory_value_rupees,

    -- 🚨 Total Alerts (ALL types except NORMAL)
    SUM(CASE WHEN alert_flag = 'ALERT' THEN 1 ELSE 0 END) AS total_alerts,

    -- 📥 Total IN Movement
    (
        SELECT SUM(quantity)
        FROM inv_ms.silver.inventory_movements
        WHERE movement_type = 'IN'
    ) AS total_in_movement,

    -- 📤 Total OUT Movement
    (
        SELECT SUM(quantity)
        FROM inv_ms.silver.inventory_movements
        WHERE movement_type = 'OUT'
    ) AS total_out_movement

FROM inv_ms.gold.inventory_alerts;

total_current_stock,total_inventory_value_rupees,total_alerts,total_in_movement,total_out_movement
375456,1.012534408E9,15,1108756,733786


**ALERT SYSTEM**

In [0]:
from pyspark.sql.functions import *

sku = spark.read.table("inv_ms.silver.sku_master")

# create threshold for all categories
threshold_df = (
    sku.select("category").distinct()
    .withColumn("critical_pct", lit(30))
    .withColumn("high_risk_pct", lit(50))
    .withColumn("warning_pct", lit(75))
)

threshold_df.write \
    .mode("overwrite") \
    .saveAsTable("inv_ms.silver.category_threshold")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import broadcast

# -------------------------
# STEP 1: Read Silver tables
# -------------------------
inv = spark.read.table("inv_ms.silver.inventory_movements")
sku = spark.read.table("inv_ms.silver.sku_master")
threshold = spark.read.table("inv_ms.silver.category_threshold")

# -------------------------
# STEP 2: Inventory balance
# -------------------------
inventory_balance = (
    inv
    .groupBy("sku_id", "warehouse_id")
    .agg(
        sum(
            when(col("movement_type") == "IN", col("quantity"))
            .otherwise(-col("quantity"))
        ).alias("on_hand_qty")
    )
)

# -------------------------
# STEP 3: Join SKU + Threshold
# -------------------------
df = (
    inventory_balance
    .join(broadcast(sku), "sku_id", "left")
    .join(broadcast(threshold), "category", "left")
)

# -------------------------
# STEP 4: Remove negative stock (Gold layer fix)
# -------------------------
df = df.withColumn(
    "on_hand_qty",
    greatest(col("on_hand_qty"), lit(0))
)

# -------------------------
# STEP 5: Handle NULLs
# -------------------------
df = df.fillna({
    "reorder_point": 1,
    "critical_pct": 30,
    "high_risk_pct": 50,
    "warning_pct": 75
})

# -------------------------
# STEP 6: Stock percentage (for dashboard)
# -------------------------
df = df.withColumn(
    "stock_pct",
    (col("on_hand_qty") / col("reorder_point")) * 100
)

# -------------------------
# STEP 7: FINAL ALERT LOGIC
# -------------------------
alerts = df.withColumn(
    "alert_level",
    
    # 🔴 Highest priority
    when(col("on_hand_qty") == 0, "OUT_OF_STOCK")
    
    # 🔴 Critical low stock
    .when(col("on_hand_qty") < col("reorder_point") * 0.3, "CRITICAL")
    
    # 🟠 High risk
    .when(col("on_hand_qty") < col("reorder_point") * 0.5, "HIGH_RISK")
    
    # 🟡 Warning
    .when(col("on_hand_qty") < col("reorder_point") * 0.75, "WARNING")
    
    # 🟢 Healthy
    .otherwise("NORMAL")
)

# -------------------------
# STEP 8: Alert flag
# -------------------------
alerts = alerts.withColumn(
    "alert_flag",
    when(col("alert_level").isin("OUT_OF_STOCK", "CRITICAL", "HIGH_RISK", "WARNING"), "ALERT")
    .otherwise("OK")
)

# -------------------------
# STEP 9: Add timestamp
# -------------------------
alerts = alerts.withColumn("alert_time", current_timestamp())

# -------------------------
# STEP 10: Save Gold table
# -------------------------
alerts.write \
    .mode("overwrite") \
    .saveAsTable("inv_ms.gold.inventory_alerts")

# -------------------------
# STEP 11: Validation
# -------------------------
print("Gold table created: inv_ms.gold.inventory_alerts")

alerts.groupBy("category", "alert_level") \
      .count() \
      .orderBy("category") \
      .show()

display(alerts)

Gold table created: inv_ms.gold.inventory_alerts
+-----------+------------+-----+
|   category| alert_level|count|
+-----------+------------+-----+
|Electronics|      NORMAL|  288|
|Electronics|     WARNING|    2|
|    Fashion|OUT_OF_STOCK|    5|
|    Fashion|     WARNING|    1|
|    Fashion|      NORMAL|  343|
|    Fashion|    CRITICAL|    1|
|    Grocery|      NORMAL|  354|
|    Grocery|OUT_OF_STOCK|    6|
+-----------+------------+-----+



category,sku_id,warehouse_id,on_hand_qty,reorder_point,unit_cost,ingestion_timestamp,source_file,critical_pct,high_risk_pct,warning_pct,stock_pct,alert_level,alert_flag,alert_time
Electronics,SKU-20,WH-7,159,38,800.64,2026-05-01T13:57:32.880Z,s3://retail-inv-ms/sku_master/sku_master.csv,30,50,75,418.42105263157896,NORMAL,OK,2026-05-05T04:51:43.559Z
Grocery,SKU-90,WH-1,434,15,4282.73,2026-05-01T13:57:32.880Z,s3://retail-inv-ms/sku_master/sku_master.csv,30,50,75,2893.3333333333335,NORMAL,OK,2026-05-05T04:51:43.559Z
Electronics,SKU-18,WH-6,155,31,2671.06,2026-05-01T13:57:32.880Z,s3://retail-inv-ms/sku_master/sku_master.csv,30,50,75,500.0,NORMAL,OK,2026-05-05T04:51:43.559Z
Grocery,SKU-34,WH-5,388,23,4625.65,2026-05-01T13:57:32.880Z,s3://retail-inv-ms/sku_master/sku_master.csv,30,50,75,1686.9565217391305,NORMAL,OK,2026-05-05T04:51:43.559Z
Electronics,SKU-2,WH-7,185,18,2553.6,2026-05-01T13:57:32.880Z,s3://retail-inv-ms/sku_master/sku_master.csv,30,50,75,1027.7777777777778,NORMAL,OK,2026-05-05T04:51:43.559Z
Grocery,SKU-19,WH-10,337,27,908.2,2026-05-01T13:57:32.880Z,s3://retail-inv-ms/sku_master/sku_master.csv,30,50,75,1248.148148148148,NORMAL,OK,2026-05-05T04:51:43.559Z
Grocery,SKU-63,WH-5,391,23,1685.81,2026-05-01T13:57:32.880Z,s3://retail-inv-ms/sku_master/sku_master.csv,30,50,75,1700.0,NORMAL,OK,2026-05-05T04:51:43.559Z
Fashion,SKU-58,WH-5,461,50,3278.05,2026-05-01T13:57:32.880Z,s3://retail-inv-ms/sku_master/sku_master.csv,30,50,75,922.0000000000001,NORMAL,OK,2026-05-05T04:51:43.559Z
Fashion,SKU-12,WH-6,258,46,1809.83,2026-05-01T13:57:32.880Z,s3://retail-inv-ms/sku_master/sku_master.csv,30,50,75,560.8695652173913,NORMAL,OK,2026-05-05T04:51:43.559Z
Fashion,SKU-67,WH-5,328,49,3483.39,2026-05-01T13:57:32.880Z,s3://retail-inv-ms/sku_master/sku_master.csv,30,50,75,669.3877551020408,NORMAL,OK,2026-05-05T04:51:43.559Z


In [0]:
alerts.groupBy("alert_level").count().show()

+------------+-----+
| alert_level|count|
+------------+-----+
|      NORMAL|  985|
|     WARNING|    3|
|OUT_OF_STOCK|   11|
|    CRITICAL|    1|
+------------+-----+



**AUDIT TRAIL HISTORY
**

In [0]:
from pyspark.sql.functions import col, max as spark_max

# -------------------------
# STEP 1: Enable Delta properties
# -------------------------
spark.sql("""
ALTER TABLE inv_ms.gold.inventory_alerts SET TBLPROPERTIES (
  delta.enableChangeDataFeed = true,
  delta.logRetentionDuration = '30 days',
  delta.deletedFileRetentionDuration = '7 days'
)
""")

print("✅ Delta audit properties enabled")

# -------------------------
# STEP 2: Get table history
# -------------------------
history_df = spark.sql("""
DESCRIBE HISTORY inv_ms.gold.inventory_alerts
""")

print("📜 Table History:")
display(history_df)

# -------------------------
# STEP 3: Create audit table
# -------------------------
audit_df = history_df.select(
    col("version"),
    col("timestamp"),
    col("operation"),
    col("operationParameters"),
    col("userName")
)

audit_df.write \
    .mode("overwrite") \
    .saveAsTable("inv_ms.gold.inventory_alerts_audit")

print("✅ Audit table created")

# -------------------------
# STEP 4: Time travel (safe version compare)
# -------------------------
latest_version = history_df.agg(spark_max("version")).first()[0]

prev_version = latest_version - 1 if latest_version > 0 else 0

df_prev = spark.read.option("versionAsOf", prev_version) \
    .table("inv_ms.gold.inventory_alerts")

df_latest = spark.read.table("inv_ms.gold.inventory_alerts")

print(f"🔁 Comparing versions: {prev_version} vs {latest_version}")
print("Previous count:", df_prev.count())
print("Latest count:", df_latest.count())

# -------------------------
# STEP 5: AUTO FIND CDF START VERSION
# -------------------------
cdf_start_version = history_df.filter(
    col("operation") == "SET TBLPROPERTIES"
).agg(spark_max("version")).first()[0]

print(f"📌 CDF starts from version: {cdf_start_version}")

# -------------------------
# STEP 6: Read Change Data Feed (SAFE)
# -------------------------
cdf_df = spark.read.format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", cdf_start_version) \
    .table("inv_ms.gold.inventory_alerts")

print("🔍 Change Data Feed (no errors):")

display(
    cdf_df.select(
        col("_change_type"),
        col("sku_id"),
        col("warehouse_id"),
        col("on_hand_qty"),
        col("alert_level"),
        col("_commit_version"),
        col("_commit_timestamp")
    )
)

# -------------------------
# STEP 7: Operation summary
# -------------------------
summary_df = audit_df.groupBy("operation").count()

print("📊 Operation Summary:")
display(summary_df)

# -------------------------
# STEP 8: Final confirmation
# -------------------------
print("✅ Full audit trail + CDF setup completed successfully")

✅ Delta audit properties enabled
📜 Table History:


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
10,2026-05-05T04:51:49.000Z,75982514874339,charanalavala1692@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true"",""delta.logRetentionDuration"":""30 days"",""delta.deletedFileRetentionDuration"":""7 days""})",null,List(4150827371904883),b8bd67a9-2503-40f3-b1a1-4d44efa5ea83,0505-044834-o7oioykr-v2n,9,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
9,2026-05-05T04:51:42.000Z,75982514874339,charanalavala1692@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableChangeDataFeed"":""true"",""delta.deletedFileRetentionDuration"":""7 days"",""delta.enableDeletionVectors"":""true"",""delta.logRetentionDuration"":""30 days""}, statsOnLoad -> true)",null,List(4150827371904883),f5d95575-c351-481e-9bb9-d037d7d364b6,0505-044834-o7oioykr-v2n,8,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 15170, numDeletionVectorsRemoved -> 0, numOutputRows -> 1000, numOutputBytes -> 15169)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
8,2026-05-04T20:28:51.000Z,75982514874339,charanalavala1692@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true"",""delta.logRetentionDuration"":""30 days"",""delta.deletedFileRetentionDuration"":""7 days""})",null,List(4150827371904883),1c2438bb-f0f9-4ae1-940d-2032c55eb5da,0504-180845-xjnpkm3i-v2n,7,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
7,2026-05-04T20:27:05.000Z,75982514874339,charanalavala1692@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true"",""delta.logRetentionDuration"":""30 days"",""delta.deletedFileRetentionDuration"":""7 days""})",null,List(4150827371904883),030d5e84-9593-4ae4-84a2-478151ce2c23,0504-180845-xjnpkm3i-v2n,6,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
6,2026-05-04T20:24:43.000Z,75982514874339,charanalavala1692@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.enableChangeDataFeed"":""true"",""delta.logRetentionDuration"":""30 days"",""delta.deletedFileRetentionDuration"":""7 days""})",null,List(4150827371904883),b92a6acd-b90b-4ba2-807a-bd0d18cebb50,0504-180845-xjnpkm3i-v2n,5,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
5,2026-05-04T20:11:58.000Z,75982514874339,charanalavala1692@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4150827371904883),68025e4f-5d24-44a5-944f-73505852e570,0504-180845-xjnpkm3i-v2n,4,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 15150, numDeletionVectorsRemoved -> 0, numOutputRows -> 1000, numOutputBytes -> 15170)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
4,2026-05-04T20:09:12.000Z,75982514874339,charanalavala1692@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4150827371904883),b830060e-47c4-4578-babd-dc1526020d44,0504-180845-xjnpkm3i-v2n,3,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 15150, numDeletionVectorsRemoved -> 0, numOutputRows -> 1000, numOutputBytes -> 15150)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
3,2026-05-04T20:07:16.000Z,75982514874339,charanalavala1692@gmail.com,CREATE OR REPLACE TABLE AS

✅ Audit table created
🔁 Comparing versions: 9 vs 10
Previous count: 1000
Latest count: 1000
📌 CDF starts from version: 10
🔍 Change Data Feed (no errors):


_change_type,sku_id,warehouse_id,on_hand_qty,alert_level,_commit_version,_commit_timestamp


📊 Operation Summary:


operation,count
CREATE OR REPLACE TABLE AS SELECT,7
SET TBLPROPERTIES,4


✅ Full audit trail + CDF setup completed successfully
